# SageMaker training — decider-train

Run these cells top to bottom. `SMOKE` in cell 2 decides everything downstream: SMOKE runs the
smoke config with a 3-step budget and never launches the real arms; the real run launches
`configs/baseline_replica.yaml` and `configs/decider_v2.yaml` at their own budgets. The two
modes keep separate state, budget fingerprints and launch prefixes, so a smoke run can never
collide with a real job.

| # | cell | what it does |
|---|------|--------------|
| 1 | build-sources | assembles the allowlist pack under `build/sagemaker_src/` (git-ignored) — the estimator's `source_dir` |
| 2 | config | region/bucket/instance, `USE_SPOT=False`, per-mode state file paths |
| 3 | install | asserts `2.247 <= sagemaker < 3` in this kernel (the v3 SDK changed this estimator surface) |
| 4 | clients | boto3 clients, role, bucket, S3 prefixes |
| 5 | helpers | budget arithmetic from the same yaml the job receives, fingerprint/idempotency, `launch_job()` |
| 6 | prepare | carves the four shards with `dataset.py prepare` |
| 7 | upload | uploads them; skips a split when S3 already holds the same ContentLength |
| 8 | tests | runs `tests/` in the repo checkout — no GPU spend before the gates pass |
| 9 | smoke | one real `train.py` job, 3 steps, g4dn — proves the pack and the channels work |
| 10 | estimate | reads the smoke job's qps from CloudWatch, Price-List rate, and refuses to launch if the real budget cannot finish inside the run window |
| 11–12 | launches | both arms via `hyperparameters={"config": yaml, "seed": 0}` |
| 13 | monitor | `describe_training_job` until terminal; downloads artifacts; fails if `final_step != max_steps` |
| 14 | evaluate | both arm checkpoints + the untrained reference on the test split via `download_eval.py` + `evaluate.py` |
| 15 | results | `runs/RESULTS.md` + accuracy/pacing charts |
| 16 | ablations | the five Tier-2 ablation configs (five more launch jobs, fingerprinted the same way) |
| 17 | cleanup | stops every dangling `InProgress` job for this notebook (no delete API — stop is terminal) |

State lives in two JSON files next to this notebook (git-ignored): `decider_v2_state_smoke.json`
and `decider_v2_state.json`. Every launch is fingerprinted by its effective budget; relaunching
the same fingerprint is refused while the old job is `InProgress` and only reused when `Completed`.


In [ ]:
# Cell 1 - the allowlist source pack under build/sagemaker_src/ (git-ignored).
#
# SageMaker unpacks source_dir into the container and runs the entry point with THAT directory
# as cwd, so everything train.py imports, its requirements.txt (the --env_check pin list it
# enforces), the config yaml the launch cells pass and the eval helper have to be inside this
# allowlist - a 25-file pack beats a 400MB tarball taped together with base64 ink. The notebook
# lives inside the repo checkout, so REPO == Path.cwd().
#
# No tests/ directory is shipped: train.py's --selftest carries its own compact gates, so the
# preflight works in a stripped pack without dragging the test suite into the container.
import shutil
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "train.py").exists():
    raise SystemExit(
        f"no train.py under {REPO} - run this notebook with its working directory set to the "
        "decider-train checkout root (the directory that contains train.py)")

BUILD = REPO / "build" / "sagemaker_src"
if BUILD.exists():
    shutil.rmtree(BUILD)

# train.py's import closure + the pins it enforces + every config the launch cells reference +
# the eval helper used downstream by cell 14.
ALLOWLIST = [
    "train.py", "network.py", "tokenization.py", "dataset.py", "logger.py", "normalize.py",
    "evaluate.py", "download_eval.py", "inference.py", "requirements.txt",
    "configs/baseline_replica.yaml", "configs/decider_v2.yaml", "configs/smoke.yaml",
    "configs/decider_v2_long.yaml",
    "configs/ablations/v2_lora_bev.yaml",
    "configs/ablations/v2_layers24.yaml",
    "configs/ablations/v2_meanpool.yaml",
    "configs/ablations/v2_derived.yaml",
    "configs/ablations/v2_stage2.yaml",
]
for rel in ALLOWLIST:
    src = REPO / rel
    if not src.exists():
        raise SystemExit(f"missing from allowlist: {rel} (REPO = {REPO})")
    dst = BUILD / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    if dst.stat().st_size == 0:
        raise SystemExit(f"empty allowlist file: {rel}")

n_pack = sum(1 for p in BUILD.rglob("*") if p.is_file())
print(f"pack built: {n_pack} files under {BUILD.relative_to(REPO)}")
for p in sorted(p.relative_to(BUILD).as_posix() for p in BUILD.rglob("*") if p.is_file()):
    print(" ", p)


In [ ]:
# Cell 2 - parameters. The kernel runs the whole notebook in SMOKE mode when SMOKE=True.
#
# SMOKE runs `configs/smoke.yaml` through the real train.py in the real container, but with a
# 3-step override instead of the config's 100 steps — everything else (pack shape, channels,
# serializer, checkpoint wiring) is exactly the real path. The real run launches the two arm
# configs at their own budgets. The two modes keep SEPARATE state files, so a smoke launch can
# never collide with a real job (fingerprint guard works within one mode's file only).
import getpass
import json

# ---- mode                                                                       ---------------
SMOKE = True                        # False for the real run
SMOKE_MAX_STEPS = 3                 # the 3-step budget override (smoke.yaml itself says 100)

# ---- infrastructure ---------------------------------------------------------------
BUCKET = "decider-v2-000000000000-us-east-1"
REGION = "us-east-1"
ROLE_ARN = ""                       # empty -> get_execution_role(); a named role otherwise
ARM_TAG = "decider_v2"              # bucket folder; the job-name form strips '_' (SM rule)

# ---- compute -----------------------------------------------------------------------
INSTANCE_TYPE = "ml.g4dn.4xlarge"   # measured: v2 ~33 qps, baseline ~25 qps on this class
INSTANCE_COUNT = 1
USE_SPOT = False                    # default OFF: without --auto_resume an interruption is silent
MAX_RUN_SECONDS = 24 * 3600         # max_run; also the max_wait ceiling when USE_SPOT
KEEP_ALIVE = 0                      # warm-pool minutes between consecutive jobs

# ---- budget governors --------------------------------------------------------------
TARGET_PRESENTATIONS = 940_032      # 14,688 steps x 64 presentations (the recipe budget)
BASELINE_VS_V2_QPS = 1.32           # baseline is 1.32x SLOWER than v2 at identical batch (measured)

# ---- local paths --------------------------------------------------------------------
REPO = Path.cwd()
DATA_DIR = REPO / ("data_smoke" if SMOKE else "data_prepared")
RUNS_LOCAL = REPO / "runs"

# ---- state -------------------------------------------------------------------------- 
STATE_PATH = REPO / ("decider_v2_state_smoke.json" if SMOKE else "decider_v2_state.json")
state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {}
state.setdefault("jobs", {})
state.setdefault("launches", {})

print(f"SMOKE={SMOKE}  bucket={BUCKET}  state={STATE_PATH.name}  instance={INSTANCE_TYPE}")
print(f"defaults: use_spot={USE_SPOT}  max_run={MAX_RUN_SECONDS}s  target={TARGET_PRESENTATIONS:,} presentations")


In [ ]:
# Cell 3 - pin the SageMaker SDK. SageMaker 3.x redesigned the Estimator surface (checkpoint
# kwargs moved onto estimator properties, hyperparameter typing changed, describe endpoints
# renamed); this notebook validates against the 2.247 constructor with inspect.signature. Its
# checkpoint kwargs are the v2 `checkpoint_s3_uri` / `checkpoint_local_path` pair - there is no
# `checkpoint_config=` kwarg on PyTorch in any 2.x. Major 3 in this kernel is a hard
# stop so the buggy path is never reached (cells fail HERE instead of three hours in).
import importlib.metadata as im
import json
import math
import os
import re
import subprocess
import sys
import time

def _require(name, lo, hi):
    """Version window check: '2.247.0' -> (2, 247); raises SystemExit with the pip fix."""
    pin = f"{name}>={lo[0]}.{lo[1]},<{hi[0]}"
    try:
        version = tuple(int(x) for x in im.version(name).split(".")[:2])
    except im.PackageNotFoundError:
        raise SystemExit(f"{name} is NOT installed in this kernel - pip install \"{pin}\"") from None
    if not (lo <= version < hi):
        raise SystemExit(f"{name} {version} outside the supported window [{lo}, {hi}) - "
                         f"pip install \"{pin}\"")
    return version

SM2 = _require("sagemaker", lo=(2, 247), hi=(3, 0))

import boto3
import sagemaker
import yaml

print("sagemaker", sagemaker.__version__, "| boto3", boto3.__version__, "| pyyaml", yaml.__version__)


In [ ]:
# Cell 4 - session, role, clients, bucket. Region is pinned on the session so a stray default
# profile cannot drag API calls to the wrong account/region pair (the classic way to "lose" a
# training job: it runs fine, the notebook silently watches the wrong endpoint).
import datetime as dt

boto_session = boto3.Session(region_name=REGION)
sagemaker_session = sagemaker.Session(boto_session=boto_session, default_bucket=BUCKET)
account = boto_session.client("sts").get_caller_identity()["Account"]
ROLE = ROLE_ARN or sagemaker.get_execution_role(sagemaker_session)

sm      = boto_session.client("sagemaker")
s3      = boto_session.client("s3")
logs    = boto_session.client("logs")                                       # filter_log_events (cell 10)
pricing = boto_session.client("pricing", region_name="us-east-1")           # the API only serves us-east-1

# bucket: create on need (us-east-1 takes no LocationConstraint), then verify access
try:
    s3.head_bucket(Bucket=BUCKET)
except s3.exceptions.NoSuchBucket:  # us-east-1 takes no LocationConstraint; others do
    kwargs = {} if REGION == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": REGION}}
    s3.create_bucket(Bucket=BUCKET, **kwargs)
    print(f"created bucket {BUCKET}")
s3.head_bucket(Bucket=BUCKET)

S3_DATA = f"s3://{BUCKET}/{ARM_TAG}/data"          # /train | /val | /calib | /test  -> questions.parquet
S3_RUNS = f"s3://{BUCKET}/{ARM_TAG}/runs"          #  output_path of the training jobs
S3_CKP  = f"s3://{BUCKET}/{ARM_TAG}/checkpoints"   #  checkpoint_s3_uri (spot restarts + the final copy)
S3_EVAL = f"s3://{BUCKET}/{ARM_TAG}/eval"          #  output_path of the evaluate.py jobs (cell 14)

RUNS_LOCAL.mkdir(exist_ok=True)

print(f"account {account}  role {ROLE}")
print(f"S3_DATA {S3_DATA}\nS3_RUNS {S3_RUNS}\nS3_CKP  {S3_CKP}\nS3_EVAL {S3_EVAL}")


In [ ]:
# Cell 5 - budget arithmetic, state helpers, the reuse guard and the launcher. Every launch cell
# below is thin; everything lives here once.
#
# The budget helper re-implements train.py's precedence (argparse < yaml < CLI) for the four
# budget-relevant arguments, so each fingerprint comes from the SAME yaml the job receives:
# change a config -> new fingerprint -> fresh job; unchanged yaml -> the guard decides.
import hashlib
import inspect as _inspect

# BUDGET_DEFAULTS mirrors train.py's argparse defaults verbatim (tests/test_core.py section (h)
# re-checks them against train.py itself, so a drift there fails CI before any launch).
BUDGET_KEYS = ["max_steps", "epochs", "batch_size", "grad_accum_steps"]
BUDGET_DEFAULTS = {"max_steps": -1, "epochs": 1, "batch_size": 32, "grad_accum_steps": 2}

# What SageMaker scrapes out of the job's stdout. One capture group each (SageMaker takes the last
# group as the value); the trailing context keeps "val_loss=" from feeding the training series.
METRIC_DEFINITIONS = [
    {"Name": "train:loss", "Regex": "loss=([0-9.]+) grad_norm="},
    {"Name": "train:acc", "Regex": "acc=([0-9.]+) lr_lora="},
    {"Name": "train:sps", "Regex": "sps=([0-9.]+)"},
    {"Name": "val:loss", "Regex": "val_loss=([0-9.]+)"},
    {"Name": "val:acc", "Regex": "val_acc=([0-9.]+)"},
]


def effective_budget(config_rel, n_train, overrides=None):
    """Budget of a run exactly as train.py builds it.

    train.py's rule (the fixed budget footgun): totals an epoch when --max_steps is unset, and
    honors --max_steps when it is set; with --max_steps set train.py ignores --epochs entirely
    and iterates epochs until total_steps, so steps_from_epochs is informational only.
    Presentations = steps x batch.
    """
    cfg = yaml.safe_load((REPO / config_rel).read_text(encoding="utf-8")) or {}
    optim = cfg.get("optim", {}) or {}
    b = {k: int(optim.get(k, BUDGET_DEFAULTS[k])) for k in BUDGET_KEYS}
    for k, v in (overrides or {}).items():
        if k not in BUDGET_DEFAULTS:
            raise KeyError(f"override {k} is not budget-relevant ({sorted(BUDGET_DEFAULTS)})")
        b[k] = int(v)
    # train.py applies the config's data.limit_train when loading the channel; mirror it or the
    # steps/epoch here is computed on rows the job never sees (smoke.yaml sets it to 2000).
    limit = int((cfg.get("data") or {}).get("limit_train") or 0)
    n_used = min(n_train, limit) if limit > 0 else n_train
    # steps_per_epoch = ceil(ceil(N / micro-batch) / accumulation): identical double ceil
    steps_per_epoch = math.ceil(math.ceil(n_used / b["batch_size"]) / b["grad_accum_steps"])
    b["steps_per_epoch"] = steps_per_epoch
    b["steps_from_epochs"] = steps_per_epoch * b["epochs"]
    b["steps"] = b["max_steps"] if b["max_steps"] > 0 else b["steps_from_epochs"]
    b["presentations"] = b["steps"] * b["batch_size"] * b["grad_accum_steps"]
    return b


def fingerprint(b):
    keys = ["max_steps", "epochs", "batch_size", "grad_accum_steps", "steps",
            "steps_per_epoch", "steps_from_epochs", "presentations"]
    payload = {k: b[k] for k in keys}
    return hashlib.sha1(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:12]


def job_prefix(arm):
    # base_job_name must be alphanumeric+hyphen (SageMaker rule): the config's ARM_TAG may carry
    # an underscore, so replace it here (the bucket folder keeps the readable underscore).
    return f"{'smoke-' if SMOKE else ''}decider-{ARM_TAG.replace('_', '-')}-{arm}"


def save_state():
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True))


def record_launch(arm, config_rel, overrides, hyper, job_name, budget):
    state["jobs"][job_name] = {
        "arm": arm, "config": config_rel,
        "hyperparameters": dict(hyper), "overrides": dict(overrides or {}),
        "fingerprint": fingerprint(budget),
        "budget": {k: budget[k] for k in ("steps", "steps_per_epoch", "steps_from_epochs",
                                          "presentations")},
        "started": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "InProgress",
        "base_job_name": job_prefix(arm),
    }
    state["launches"].setdefault(arm, []).append(job_name)
    save_state()
    print(f"[launch] {arm}: {job_name}")
    print(f"         fingerprint={fingerprint(budget)}  "
          f"budget={budget['steps']:,} steps x {budget['batch_size']}x{budget['grad_accum_steps']}"
          f" = {budget['presentations']:,} presentations")


def guard_reuse(arm, fp):
    """None -> do a fresh launch; else the completed job name to reuse. Never duplicate.

    Lifecycle is checked against AWS (describe_training_job), not the state file: the state
    file only remembers which job belongs to which arm and budget.
    """
    for name, meta in state["jobs"].items():
        if meta.get("arm") != arm:
            continue
        live = sm.describe_training_job(TrainingJobName=name)["TrainingJobStatus"]
        meta["status"] = live
        if live == "InProgress" and meta.get("fingerprint") == fp:
            raise SystemExit(
                f"{arm}: job {name} is already InProgress with the same budget fingerprint {fp}. "
                f"The guard refuses a duplicate. To force a fresh run, stop/delete the job in "
                f"SageMaker and remove its entry from {STATE_PATH.name}.")
        if live == "Completed" and meta.get("fingerprint") == fp:
            print(f"[reuse] {arm}: completed job {name} already has fingerprint {fp}; not re-launching")
            save_state()
            return name
    return None


# ------------------------------------------------------------------ estimator kwargs check
_sig_cache = None

def check_estimator_kwargs(kwargs, label="PyTorch"):
    """Every launched kwarg must be a NAMED parameter of the real SDK constructor.

    PyTorch.__init__ (and the base Estimator) accept **kwargs, so a typo would be swallowed
    silently and the job would run with the DEFAULT for that argument - e.g. no checkpoints,
    no spot cap. Failing here costs nothing; failing after 12k steps costs the run window.
    """
    global _sig_cache
    if _sig_cache is None:
        _sig_cache = _inspect.signature(sagemaker.estimator.PyTorch.__init__)
    named = set(_sig_cache.parameters)
    orphans = [k for k in kwargs if k not in named]
    if orphans:
        raise SystemExit(
            f"{label}: kwargs not in sagemaker.estimator.PyTorch.__init__ (sagemaker "
            f"{sagemaker.__version__}) -> {orphans}. The SDK surface moved (or a typo) - the "
            "**kwargs catch-all would have run the job with the default for these instead of "
            "raising. Check sagemaker release notes and fix the launcher.")
    return True


# ------------------------------------------------------------------ the launcher
def launch_job(arm, config_rel, n_train, overrides=None, seed=0, wait=False):
    """Create the PyTorch estimator, validate its kwargs, record the launch in the state file.

    Fixed vs the old argv serialization:
      * hyperparameters = {"config": yaml, "seed": 0, ...}: one hyperparameter per train.py
        argparse argument. No "sagemaker_args" string (that escape hatch stays in train.py but
        the notebook never uses it - shell-serialized argv is where old silent quirk bugs lived).
      * every estimator kwarg goes through check_estimator_kwargs BEFORE fit().
      * the job name comes from est.latest_training_job.name, never from estimator internals.
    """
    budget = effective_budget(config_rel, n_train, overrides)
    reused = guard_reuse(arm, fingerprint(budget))
    if reused:
        return None

    hyper = {"config": config_rel, "seed": str(seed)}
    for k, v in (overrides or {}).items():
        if k in BUDGET_DEFAULTS:
            hyper[k] = str(v)
    if USE_SPOT and hyper.get("auto_resume") != "1":
        raise SystemExit(
            "USE_SPOT=True without auto_resume=1: a spot interruption silently restarts from "
            "zero without checkpoints + auto-resume, which is exactly the failure the config "
            "was written to avoid. Set hyper['auto_resume']='1' or USE_SPOT=False.")
    if hyper.get("auto_resume") == "1" and not USE_SPOT:
        print("[note] auto_resume=1 requested without USE_SPOT: it is harmless (no interruption "
              "-> nothing to resume), but unusual - double-check the intent.")

    base = job_prefix(arm)
    assert "_" not in base and job_prefix(arm) == base, base
    ckpt_uri = f"{S3_CKP}/{base}"

    est_kwargs = dict(
        entry_point="train.py",
        source_dir=str(BUILD),             # the allowlist pack from cell 1, NOT the repo root
        role=ROLE,
        framework_version="2.7.1", py_version="py312",
        instance_type=INSTANCE_TYPE, instance_count=INSTANCE_COUNT,
        sagemaker_session=sagemaker_session,
        output_path=S3_RUNS,
        hyperparameters=hyper,
        # The v2 estimator takes these two kwargs and builds the API's CheckpointConfig={S3Uri,
        # LocalPath} from them. There is no `checkpoint_config=` kwarg on PyTorch (that is the
        # v3 ModelTrainer / the API field name); **kwargs would swallow it, and
        # check_estimator_kwargs below rejects it instead of launching a job that never syncs.
        checkpoint_s3_uri=ckpt_uri,               # /opt/ml/checkpoints -> S3_CKP/<base>
        checkpoint_local_path="/opt/ml/checkpoints",
        # MetricDefinitions: SageMaker scrapes the job's stdout for these. Without them the
        # console shows no charts and the training job page reports "No metrics". The regexes are
        # anchored on the surrounding tokens because the log line is
        #   step=N loss=... grad_norm=... acc=... lr_lora=... lr_head=... sps=...
        # and a bare "loss=" would also match "val_loss=".
        metric_definitions=METRIC_DEFINITIONS,
        # The model artifacts here are a LoRA-merged checkpoint plus tokenizer, and tarring them
        # adds minutes to every job for no benefit. The container writes them uncompressed.
        disable_output_compression=True,
        use_spot_instances=USE_SPOT,
        max_run=MAX_RUN_SECONDS,
        max_wait=MAX_RUN_SECONDS if USE_SPOT else None,
        keep_alive_period_in_seconds=int(KEEP_ALIVE * 60),
        base_job_name=base,
    )
    check_estimator_kwargs(est_kwargs)

    est = sagemaker.estimator.PyTorch(**est_kwargs)
    est.fit(inputs={"train": f"{S3_DATA}/train", "val": f"{S3_DATA}/val"},
            wait=wait, logs="None")        # no dead first-run eyeballing: the monitor cell owns tailing
    name = est.latest_training_job.name     # raises if the job never started
    record_launch(arm, config_rel, overrides, hyper, name, budget)
    print(f"         ckpt prefix {ckpt_uri}/{name}")
    return est


# ------------------------------------------------------------------ CloudWatch throughput
def read_job_qps(job_name, minutes_back=90):
    """Steady-state `sps` (presentations/second) from the job's CloudWatch log stream.

    filter_log_events (NOT describe: the rate meter should not pay for job descriptions), and
    only the LAST few windows: the first window mixes in worker start-up, steady state is the
    tail (measured: the first window reads ~5x low on cold CUDA compile).
    """
    start = dt.datetime.now(dt.timezone.utc) - dt.timedelta(minutes=minutes_back)
    samples = []
    for page in logs.get_paginator("filter_log_events").paginate(
            logGroupName="/aws/sagemaker/TrainingJobs", logStreamNamePrefix=job_name,
            startTime=int(start.timestamp() * 1000)):
        for ev in page.get("events", []):
            m = re.search(r"\bsps=([0-9]+(?:\.[0-9]+)?)", ev["message"])
            if m:
                samples.append((ev["timestamp"], float(m.group(1))))
    if not samples:
        return None
    tail = [v for _t, v in samples[-5:]]
    return {"samples": len(samples),
            "median": sorted(tail)[len(tail) // 2],
            "steady": max(tail),
            "first": samples[0]}


# ------------------------------------------------------------------ Price List
def price_per_hour(instance_type, region=REGION):
    """USD/h for an AmazonSageMaker instance-hour; None when the API is unreachable/empty.

    Then prints NOT RESOLVED: a pricing failure must never invent a number and must never kill
    the run - the budget verdict in cell 10 falls back to the measured-rate path.
    """
    filters = [{"Type": "TERM_MATCH", "Field": "ServiceCode", "Value": "AmazonSageMaker"},
               {"Type": "TERM_MATCH", "Field": "instanceType", "Value": instance_type},
               {"Type": "TERM_MATCH", "Field": "regionCode", "Value": region},
               {"Type": "TERM_MATCH", "Field": "operatingSystem", "Value": "Linux"}]
    try:
        for page in pricing.get_paginator("get_products").paginate(ServiceCode="AmazonSageMaker", Filters=filters):
            for item in page.get("PriceList", []):
                rec = json.loads(item)
                attrs = rec.get("product", {}).get("attributes", {})
                if region != attrs.get("regionCode"):
                    continue                 # get_products honors the filter, but be strict anyway
                demand = rec.get("terms", {}).get("OnDemand", {})
                for offer in demand.values():
                    for dim in offer.get("priceDimensions", {}).values():
                        rate = dim.get("pricePerUnit", {}).get("USD")
                        if rate and dim.get("unit", "").lower().startswith("hour"):
                            return float(rate), dim.get("description", "")
    except Exception as e:  # noqa: BLE001 - pricing is best-effort by design (NO hard dependency)
        print(f"pricing resolution failed ({type(e).__name__}: {e})")
        print("NOT RESOLVED - Price List response unusable for this region; "
              "cell 10 falls back to the measured-rate budget verdict.")
        return None, ""
    print("NOT RESOLVED - no AmazonSageMaker OnDemand offer matched "
          f"(instanceType {instance_type}, regionCode {region}, operatingSystem Linux)")
    return None, ""


In [ ]:
# Cell 6 - carve the four shards with dataset.py prepare. Re-runs are cheap: dataset.py reuses
# an existing carve when the manifest's seed/calib_target match, and caps only bite on a fresh
# write (SMOKE caps: 4000 train / 400 val / 400 calib / 4000 test -> a Git-LFS-sized shard set
# that still has every task/domain present in the mix).
import json
import os
import subprocess

cap_args = []
if SMOKE:
    cap_args = ["--smoke_train", "4000", "--smoke_val", "400",
                "--smoke_calib", "400", "--smoke_test", "4000"]
cmd = [sys.executable, "dataset.py", "prepare", "--out", str(DATA_DIR),
       "--seed", "0", "--calib_target", "20000", "--val_frac", "0.03", *cap_args]
env = {**os.environ, "PYTHONIOENCODING": "utf-8", "TOKENIZERS_PARALLELISM": "false"}
print(" ".join(os.path.basename(c) if isinstance(c, str) else c for c in cmd))
proc = subprocess.run(cmd, cwd=str(REPO), env=env)
if proc.returncode != 0:
    raise SystemExit("dataset.py prepare failed")

manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
# sizes, not stats: stats describes the FULL exploded train split, while the shard train.py
# actually trains on is the carved (and, in SMOKE, capped) one. steps/epoch must mirror the
# parquet the job receives or the fingerprint's arithmetic is off.
N_TRAIN = manifest["sizes"]["train"]
print(f"\nN_TRAIN={N_TRAIN:,} (written train shard; drives every budget's steps/epoch math)")
for k in ("train", "val", "calib", "test"):
    assert (DATA_DIR / k / "questions.parquet").exists(), f"missing shard: {k}"


In [ ]:
# Cell 7 - upload the four shards, skipping any PUT whose destination object already carries
# the same ContentLength (idempotent re-runs do not resend the train shard).
uploaded, skipped = [], []
SHARD_KEYS = {k: f"{ARM_TAG}/data/{k}/questions.parquet" for k in ("train", "val", "calib", "test")}
for split, key in SHARD_KEYS.items():
    path = DATA_DIR / split / "questions.parquet"
    size = path.stat().st_size
    try:
        head = s3.head_object(Bucket=BUCKET, Key=key)
        if head["ContentLength"] == size:
            skipped.append(split)
            continue
        print(f"{split}: S3 has {head['ContentLength']}B, local has {size}B - re-uploading")
    except s3.exceptions.NoSuchKey:
        pass
    s3.upload_file(str(path), BUCKET, key)
    uploaded.append(split)

assert set(SHARD_KEYS) == set(uploaded) | set(skipped), (uploaded, skipped)
print(f"uploaded {uploaded}, skipped (same ContentLength) {skipped}")
print("data URI convention:", ", ".join(f"{S3_DATA}/{k}" for k in SHARD_KEYS))


In [ ]:
# Cell 8 - run the full test battery BEFORE spending any GPU minutes. The kernel's own python
# runs tests/: they need the pinned stack (torch/transformers/peft/datasets) in THIS kernel
# env, which Studio kernels do not ship by default - pip-install them lazily exactly the way
# train.py's --env_check would (same pins from requirements.txt; torch must already be here).
import importlib
import os
import subprocess
import sys

def _need(module):
    try:
        importlib.import_module(module)
        return None
    except Exception as e:  # noqa: BLE001 - any import trouble counts (partial wheels too)
        return f"{type(e).__name__}: {e}"

missing = [m for m in ("torch", "transformers", "peft", "datasets") if (fail := _need(m))]
if missing:
    print("stack incomplete in this kernel env ->", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        "-r", str(REPO / "requirements.txt")],
                       cwd=str(REPO), capture_output=True, text=True)
    print(r.stdout[-2000:] + r.stderr[-2000:])
    still_missing = [m for m in "torch transformers peft datasets".split() if _need(m)]
    if still_missing:
        raise SystemExit(f"pip install did not fix the stack - still missing {still_missing}")

# test_core.py is a script with its own main() (run with --device cpu here: the kernel has no GPU).
# `pytest tests` would instead collect its test_* helpers and fail on their `device` argument.
cmd = [sys.executable, "tests/test_core.py", "--device", "cpu"]
r = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, "PYTHONIOENCODING": "utf-8"},
                   capture_output=True, text=True)
print(r.stdout[-4000:] + r.stderr[-1500:])
if r.returncode != 0:
    raise SystemExit("tests failed - do NOT launch (fix locally first; the GPU run just burns money)")
print("tests: PASS")


In [ ]:
# Cell 9 - the SMOKE run: configs/smoke.yaml with a 3-step budget override through the REAL
# train.py in the REAL container (pack, channels, serializer, checkpoint wiring are all the
# production path). Finishes in a couple of minutes on a g4dn; the qps tail feeds cell 10.
# In real mode this cell prints a skip - the two arm launches own the GPU time.
overrides = {"max_steps": SMOKE_MAX_STEPS} if SMOKE else None
SMOKE_JOB = None
if not SMOKE:
    print("[skip] SMOKE=False: cells 11-12 launch the real arms; cell 9 is the smoke path only")
else:
    est_smoke = launch_job("smoke", "configs/smoke.yaml", n_train=N_TRAIN,
                           overrides=overrides, seed=0, wait=True)
    if est_smoke is not None:
        SMOKE_JOB = est_smoke.latest_training_job.name
    else:                                # guard_reuse re-used a completed job -> recover its name
        for name, meta in state["jobs"].items():
            if meta.get("arm") == "smoke":
                SMOKE_JOB = name
                break
    if not SMOKE_JOB:
        raise SystemExit("no smoke job available (fresh launch refused and nothing reusable recorded)")
    print(f"SMOKE_JOB={SMOKE_JOB}")


In [ ]:
# Cell 10 - the runtime and price verdict for the REAL budget, built from evidence in this
# order: (1) the smoke job's own sps tail from CloudWatch (filter_log_events), (2) the measured
# g4dn ratio baseline = v2 x 1.32 applied to it, (3) the Price List on-demand rate (NOT RESOLVED
# is a legal outcome - the verdict then rides on 1+2 only). The guard raises BEFORE cells 11-12
# when the real budget cannot finish inside MAX_RUN_SECONDS; in SMOKE mode it prints the verdict
# and continues (a 3-step smoke run cannot hit the guard by construction).
if not SMOKE:
    raise SystemExit("cell 10 is the smoke-derived estimate; it is a no-op in real mode - "
                     "the real launches happen in cells 11-12 without any smoke run")

base_b = effective_budget("configs/baseline_replica.yaml", N_TRAIN)
v2_b = effective_budget("configs/decider_v2.yaml", N_TRAIN)
print(f"real budgets: baseline {base_b['steps']:,} steps ({base_b['presentations']:,} presentations) | "
      f"v2 {v2_b['steps']:,} steps ({v2_b['presentations']:,} presentations)")
print(f"steps/epoch at N_TRAIN={N_TRAIN:,}: baseline {base_b['steps_per_epoch']:,} | "
      f"v2 {v2_b['steps_per_epoch']:,} (--max_steps sets the budget; --epochs does not cap it)")

q = read_job_qps(SMOKE_JOB)
if q is None:
    raise SystemExit(
        f"no sps=... windows found for {SMOKE_JOB}. The smoke job's log stream never reached "
        "CloudWatch (or it logged no training steps) - open the job's log in the SageMaker "
        "console and re-check the launch before continuing.")
qps_v2, qps_first = q["steady"], q["first"][1]
qps_base = qps_v2 / BASELINE_VS_V2_QPS   # baseline is 1.32x SLOWER (identical batch, measured)
print(f"[throughput] {SMOKE_JOB}: first window {qps_first:.2f} sps -> steady {qps_v2:.2f} sps "
      f"({q['samples']} windows; reference points from the g4dn.4xlarge plate: v2 33.3, baseline 25.2)")

h_base = base_b["presentations"] / qps_base / 3600
h_v2 = v2_b["presentations"] / qps_v2 / 3600
price, price_note = price_per_hour(INSTANCE_TYPE)
if price is not None:
    print(f"[price] {price_note or 'AmazonSageMaker'}: {price:.3f} USD/h -> "
          f"baseline {h_base * price * INSTANCE_COUNT:.0f} USD "
          f"+ v2 {h_v2 * price * INSTANCE_COUNT:.0f} USD (spot discount NOT applied)")
else:
    print("[price] unavailable: budget verdict rides on evidence 1+2 only")

total_h = (h_base + h_v2) * INSTANCE_COUNT
verdict_ok = total_h * 3600 <= MAX_RUN_SECONDS
print(f"[estimate] baseline {h_base:.2f} h + v2 {h_v2:.2f} h = {total_h:.2f} h vs window "
      f"{MAX_RUN_SECONDS / 3600:.2f} h -> {'FITS' if verdict_ok else 'DOES NOT FIT'}")
print("[caveat] the qps above comes from a 3-step smoke run on the smoke config - cross-check it "
      "against the g4dn plate (33.3/25.2 sps) before trusting any USD figure.")
if not verdict_ok:
    if SMOKE:
        print("[WARN] SMOKE mode: the guard would raise here on a real budget; continuing "
              "(a smoke run cannot hit this guard by construction)")
    else:
        raise SystemExit(
            f"the real budget does NOT fit the window: {total_h:.2f} h needed vs "
            f"{MAX_RUN_SECONDS / 3600:.2f} h available. Do NOT launch cells 11-12 until this fits; "
            "fix with a smaller budget, a faster instance class, or spot + auto_resume.")


In [ ]:
# Cell 11 - launch the baseline arm (configs/baseline_replica.yaml).
#
# In real mode: non-blocking launch (wait=False, logs="None") - cell 13 owns polling, and the
# v2 job goes up in parallel from cell 12. In SMOKE mode: cells 13-14 only prove plumbing, so
# this launch runs with a tiny override budget and blocking wait (the artifact layout is
# exercised immediately). In both modes the launch is fingerprinted by its EFFECTIVE budget -
# the launcher computes it from the SAME yaml the job receives.
overrides = {"max_steps": SMOKE_MAX_STEPS} if SMOKE else None
bh = effective_budget("configs/baseline_replica.yaml", N_TRAIN, overrides)
print(f"baseline budget: fingerprint={fingerprint(bh)} -> {bh}")
est_base = launch_job("baseline", "configs/baseline_replica.yaml", n_train=N_TRAIN,
                      overrides=overrides, seed=0, wait=SMOKE)
if est_base is None:
    print("[reuse] baseline: nothing to watch here - cell 13 loops over recorded jobs (fresh or old)")


In [ ]:
# Cell 12 - launch the v2 arm (configs/decider_v2.yaml), parallel to the baseline in real mode.
#
# Same mechanics: fingerprint-from-config guard, checkpoint wiring, kwargs validation.
overrides = {"max_steps": SMOKE_MAX_STEPS} if SMOKE else None
bh = effective_budget("configs/decider_v2.yaml", N_TRAIN, overrides)
print(f"v2 budget: fingerprint={fingerprint(bh)} -> {bh}")
est_v2 = launch_job("v2", "configs/decider_v2.yaml", n_train=N_TRAIN,
                    overrides=overrides, seed=0, wait=SMOKE)
if est_v2 is None:
    print("[reuse] v2: nothing new to watch here - cell 13 polls every recorded InProgress job")


In [ ]:
# Cell 13 - the monitor: poll describe_training_job for every recorded job that is InProgress or
# Stopping, until each is terminal; then download its artifacts under runs/<job>/ with the
# sketchy-file skip (a file already on disk with the same ContentLength is not re-downloaded);
# then CHECK final_step vs budget. A Completed run that ends off-budget fails HERE with a
# diagnostic and the receipt of where to look - that is the manager cell's whole reason to exist.
# SageMaker has no delete API: Stopping -> Stopped is the terminal form of "killed it ourselves".
import datetime

POLL_SECONDS = 60

def describe(name):
    return sm.describe_training_job(TrainingJobName=name)

def key_prefix(uri):
    """s3://bucket/key/... -> key/...   list_objects_v2 takes a KEY prefix, not a URI."""
    if uri.startswith("s3://"):
        bucket, _sep, key = uri[len("s3://"):].partition("/")
        assert bucket == BUCKET, f"prefix bucket {bucket} != {BUCKET}"
        return key
    return uri

def download_prefix(prefix, local_dir):
    """Copy every object under prefix to local_dir (skip: present + same size). Returns (new, total)."""
    prefix = key_prefix(prefix)
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix):
        keys.extend(page.get("Contents", []))
    n_new = 0
    base = prefix.rstrip("/")
    for obj in keys:
        rel = obj["Key"][len(base):].lstrip("/")
        dst = local_dir / rel
        if dst.exists() and dst.stat().st_size == obj["Size"]:
            continue
        dst.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(BUCKET, obj["Key"], str(dst))
        n_new += 1
    return n_new, len(keys)

pending = [n for n in state["jobs"]
           if describe(n)["TrainingJobStatus"] in ("InProgress", "Stopping")]

while pending:
    for name in list(pending):
        d = describe(name)
        status = d["TrainingJobStatus"]
        state["jobs"][name].update({
            "status": status,
            "secondary_status": d.get("SecondaryStatus", ""),
            "checked": datetime.datetime.now().isoformat(timespec="seconds"),
        })
        if status == "Failed":
            print(f"[FAIL] {name}: {d.get('FailureReason', '')}")
            save_state()
            raise SystemExit(f"job {name} Failed - see its CloudWatch log; the run cannot continue")
        if status in ("Completed", "Stopped"):
            pending.remove(name)
    save_state()
    if not pending:
        break
    print("waiting:", ", ".join(n for n in pending), "----------------------")
    time.sleep(POLL_SECONDS)

# downloads + the final_step check, per terminal job
TERMINALS = [n for n, meta in state["jobs"].items()
             if describe(n)["TrainingJobStatus"] in ("Completed", "Stopped", "Failed")]
for name in sorted(TERMINALS):
    meta = state["jobs"][name]
    # Read the live status: the polling loop above only revisits jobs it saw as InProgress, so a
    # job that failed between launch and this pass still carries its recorded "InProgress" here.
    live = describe(name)["TrainingJobStatus"]
    meta["status"] = live
    if meta["arm"].startswith("eval-"):
        print(f"[eval] {name}: eval job - its result json is pulled inside cell 14")
        continue
    local = RUNS_LOCAL / name
    new_out, tot_out = download_prefix(f"{S3_RUNS}/{name}/output/", local / "output")
    new_ckpt, tot_ckpt = download_prefix(f"{S3_CKP}/{meta['base_job_name']}/{name}/checkpoints/", local / "checkpoints")
    print(f"[download] {name}/{meta['arm']}: output {new_out}/{tot_out} new, ckpt {new_ckpt}/{tot_ckpt} new")

    summary_path = local / "output" / "data" / "summary.json"
    if live == "Stopped":
        print(f"[stopped] {name}: partial run (our own stop or spot) - artifacts kept, no budget verdict")
        continue
    if live == "Failed":
        print(f"[FAIL] {name}: {describe(name).get('FailureReason', '')} - the job failed before writing "
              "summary.json; read its CloudWatch log (preflight/argv errors land there first)")
        continue
    if not summary_path.exists():
        raise SystemExit(f"NOT RESOLVED: {name} says Completed but runs/{name}/output/data/summary.json "
                         "did not download - check the S3 output prefix and this kernel's S3 permissions")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    final_step, total_steps = summary.get("final_step"), summary.get("total_steps")
    expected = meta["budget"]["steps"]
    if final_step != expected:
        raise SystemExit(
            f"NOT RESOLVED: {name} ended off-budget - final_step={final_step} vs "
            f"total_steps={total_steps} (state budget says {expected}). The job says Completed, so "
            "this is NOT a crash: likely the epochs cap truncated it (cell 5 warned), or the "
            "config changed mid-run. Inspect runs/<job>/output/data/metrics.jsonl tail and the "
            "CONFIG banner in the CloudWatch log before re-checking the verifier.")
    print(f"[verify] {name}: final_step={final_step} == budget {expected}  best_val_acc="
          f"{summary.get('best_val_acc')}")
save_state()


In [ ]:
# Cell 14 - evaluate BOTH arm checkpoints and the untrained reference on the test split.
#
# Each eval is a real SageMaker job running download_eval.py (which pulls the checkpoint prefix
# through the container's own creds, then shells out to evaluate.py). The eval hyperparameters
# are all evaluate.py argparse names - env_check=1 so evaluate.py pins the container libraries
# exactly like train.py does (transformers/peft are NOT preinstalled in the framework image).
# Outputs land at S3_EVAL/<job>/output/data/<tag>.json and are pulled to runs/evals/<tag>.json.
REFERENCE_ID = "avbiswas/bev-decider-0.4B"   # published v1 recipe, no fine-tune (the un-trained lamp)

EVAL_FLAGS = ["checkpoint", "reference", "test_channel", "calib_channel", "out", "tag", "env_check",
              "limit", "batch_size", "invariance", "invariance_examples", "invariance_shuffles",
              "predictions_out", "max_state_tokens", "max_choice_tokens", "mode", "pool"]

def launch_eval(tag, checkpoint_uri, cfg=None, limit="1600"):
    """One evaluate.py job; returns its job name (no body-side prediction files needed)."""
    cfg = cfg or f"configs/{'smoke' if SMOKE and tag != 'reference' else 'decider_v2'}.yaml"
    data = (yaml.safe_load((REPO / cfg).read_text(encoding="utf-8")) or {}).get("data", {})
    hyper = {"checkpoint": str(checkpoint_uri or ""), "tag": tag,
             "limit": str(limit), "batch_size": "8",
             "invariance": "1", "invariance_examples": "32", "invariance_shuffles": "32",
             "env_check": "1",
             "mode": str(data.get("mode", "bev")), "pool": str(data.get("pool", "bev")),
             "max_state_tokens": str(data.get("max_state_tokens", 1024)),
             "max_choice_tokens": str(data.get("max_choice_tokens", 64))}
    if tag == "reference":
        # evaluate.py's published-model path: the checkpoint is empty and --reference names the repo
        hyper["reference"] = REFERENCE_ID
    unknown = [k for k in hyper if k not in EVAL_FLAGS]
    if unknown:
        raise SystemExit(f"eval hyperparameters are not evaluate.py argv ({unknown})")

    base = job_prefix(f"eval-{tag}")
    est_kwargs = dict(
        entry_point="download_eval.py", source_dir=str(BUILD), role=ROLE,
        framework_version="2.7.1", py_version="py312",
        instance_type=INSTANCE_TYPE, instance_count=1,
        sagemaker_session=sagemaker_session,
        output_path=S3_EVAL, hyperparameters=hyper,
        # An eval job reads its checkpoint from S3 and writes no checkpoint of its own, so it needs
        # no checkpoint_s3_uri/local_path pair (and `checkpoint_config` is not a v2 kwarg).
        use_spot_instances=False, max_run=2 * 3600, max_wait=None,
        keep_alive_period_in_seconds=0, base_job_name=base)
    check_estimator_kwargs(est_kwargs, label="download_eval")
    est = sagemaker.estimator.PyTorch(**est_kwargs)
    est.fit(inputs={"test": f"{S3_DATA}/test", "calib": f"{S3_DATA}/calib"}, wait=True, logs="None")
    name = est.latest_training_job.name
    state["jobs"][name] = {"arm": f"eval-{tag}", "config": cfg, "hyperparameters": hyper,
                           "started": time.strftime("%Y-%m-%d %H:%M:%S"), "status": "Completed",
                           "base_job_name": base}
    save_state()
    prefix = f"{S3_EVAL}/{name}/output/data/"
    local = RUNS_LOCAL / "evals" / f"{tag}.json"
    local.parent.mkdir(parents=True, exist_ok=True)
    res = [o["Key"] for page in s3.get_paginator("list_objects_v2").paginate(
        Bucket=BUCKET, Prefix=key_prefix(prefix))
           for o in page.get("Contents", []) if o["Key"].endswith(".json")]
    if len(res) != 1:
        print(f"NOT RESOLVED: expected exactly one result json under {prefix}, found {res}")
        return name
    s3.download_file(BUCKET, res[0], str(local))
    d = json.loads(local.read_text(encoding="utf-8"))
    print(f"[eval:{tag}] {', '.join(f'{k}={v}' for k, v in d.items() if not isinstance(v, dict))}")
    return name


# the arm checkpoint URIs: S3_CKP/<base>/<TrainingJobName>/checkpoints/best (the CheckpointConfig
# mirror SM writes during the run; 'best' because checkpoint_selection picked best VALIDATION acc)
def best_uri(job_name):
    meta = state["jobs"][job_name]
    return f"{S3_CKP}/{meta['base_job_name']}/{job_name}/checkpoints/best"

launch_eval("reference", checkpoint_uri="", cfg="configs/decider_v2.yaml")

def completed(arm):
    out = []
    for n in state["launches"].get(arm, []):
        if describe(n)["TrainingJobStatus"] == "Completed":
            out.append(n)
    return out

if SMOKE:
    launch_eval("smoke-arm", checkpoint_uri=best_uri(SMOKE_JOB), cfg="configs/smoke.yaml")
else:
    for arm, cfg in (("baseline", "configs/baseline_replica.yaml"),
                     ("v2", "configs/decider_v2.yaml")):
        pool = completed(arm)
        if not pool:
            print(f"[skip] {arm} eval: no Completed training job (cells 11-13 launch + monitor first)")
            continue
        launch_eval(arm, checkpoint_uri=best_uri(pool[-1]), cfg=cfg)


In [ ]:
# Cell 15 - benchmark spot-checks against the two SVM harnesses (jevbench / sysone-bench), on the
# monitor-downloaded checkpoints. NOT RUN is a legal outcome: each harness needs its own clone
# checkout (JEVBENCH_ROOT / SYSONE_BENCH_ROOT) that the notebook cannot conjure; the SageMaker
# run itself is not benchmark-harness-dependent, so a missing clone never blocks the launchers.
import os
import subprocess

HARNESSES = [("jevbench", "JEVBENCH_ROOT"), ("sysone_bench", "SYSONE_BENCH_ROOT")]
BENCH_LIMIT = 64        # spot-check scale; a full sweep is a separate machine-sized job

def local_ckpt(arm):
    for job in state["launches"].get(arm, []):
        d = RUNS_LOCAL / job / "checkpoints" / "best"
        if d.is_dir() and (d / "meta.json").exists():
            return d
    return None

for harness, root_env in HARNESSES:
    script = REPO / "benchmarks" / f"run_{harness}.py"
    if not script.exists():
        print(f"[NOT RUN] {harness}: benchmarks/run_{harness}.py is not in this checkout")
        continue
    root = os.environ.get(root_env, "")
    if not (root and os.path.isdir(root)):
        print(f"[NOT RUN] {harness}: {root_env} is not set to a clone checkout")
        continue
    for arm in ("baseline", "v2") if not SMOKE else ("smoke",):
        ckpt = local_ckpt(arm)
        if ckpt is None:
            print(f"[skip] {harness}/{arm}: no downloaded checkpoint (run cells 11-13 first)")
            continue
        out = RUNS_LOCAL / f"{harness}-{arm}-limit{BENCH_LIMIT}.json"
        cmd = [sys.executable, str(script), "--checkpoint", str(ckpt),
               "--out", str(out), "--limit", str(BENCH_LIMIT),
               "--batch_size", "8", "--progress_every", "32"]
        if harness == "jevbench":
            cmd += ["--jevbench_root", root]
        if harness == "sysone_bench":
            cmd += ["--sysone_bench_root", root]
        print(f"[bench] {harness}/{arm}: {' '.join(cmd)}")
        r = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, "PYTHONIOENCODING": "utf-8"})
        print(f"[bench] {harness}/{arm}: rc={r.returncode}"
              + ("" if r.returncode == 0 else " - see the traceback above; result json absent"))


In [ ]:
# Cell 16 - RESULTS.md (runs/RESULTS.md) + the pacing chart. Both are driven by the
# monitor-downloaded artifacts, so the cell reports an honest table for whatever ran here
# plus .NOT-RUN marks for every missing arm: no cell ever invents a number.
import datetime

rows, evals = [], {}
for job, meta in sorted(state["jobs"].items()):
    if meta["arm"].startswith("eval-"):
        evals.setdefault(meta["arm"].removeprefix("eval-"), meta)
        continue
    s = RUNS_LOCAL / job / "output" / "data" / "summary.json"
    if not s.exists():
        rows.append((meta["arm"], job, "-", "FAILED/partial"))
        continue
    d = json.loads(s.read_text(encoding="utf-8"))
    rows.append((meta["arm"], job, f"{d.get('best_val_acc', float('nan'))*100:.2f}%",
                 f"{d.get('final_step')}/{meta['budget']['steps']}"))

md = [f"## SageMaker run {datetime.date.today().isoformat()}",
      f"instance {INSTANCE_TYPE}" + (" x spot" if USE_SPOT else " (on-demand)"),
      "",
      "| arm | job | best val acc | budget |",
      "|-----|-----|-------------:|--------|"]
for arm, job, acc, b in rows:
    md.append(f"| {arm} | {job} | {acc} | {b} |")
for tag in ("baseline", "v2", "reference"):
    if tag in evals:
        md.append(f"\n### eval / {tag}: *summary json preserved; read it from runs/evals/{tag}.json*")

out = RUNS_LOCAL / "RESULTS"
out.mkdir(exist_ok=True)
(out / "RESULTS.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("\n".join(md))
print(f"\nwrote {out / 'RESULTS.md'}")

# pacing chart: sps per step for every completed arm against one figure
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    print("[charts] matplotlib unavailable in this kernel - RESULTS.md still written")
    plt = None

if plt:
    fig, ax = plt.subplots(figsize=(10, 5))
    drew = False
    for job, meta in sorted(state["jobs"].items()):
        if meta["arm"].startswith("eval-"):
            continue
        p = RUNS_LOCAL / job / "output" / "data" / "metrics.jsonl"
        if not p.exists():
            continue
        steps, sps = [], []
        for line in p.read_text(encoding="utf-8").splitlines():
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(d.get("step"), int) and d.get("sps"):
                steps.append(d["step"]); sps.append(d["sps"])
        if steps:
            ax.plot(steps, sps, marker="o", markersize=3, label=meta["arm"])
            drew = True
    if drew:
        ax.set_xlabel("step"); ax.set_ylabel("presentations / second (sps)")
        ax.set_title("training pacing by arm")
        ax.legend(); ax.grid(True, alpha=0.3)
        chart = out / "pacing.png"
        fig.savefig(chart, dpi=100, bbox_inches="tight")
        print(f"wrote {chart}")
    else:
        print("[charts] no metrics.jsonl downloaded - nothing to plot")


In [ ]:
# Cell 17 - the Tier-2 ablation ladder (five configs, one job each). Every rule from cells 11-12
# applies: config-driven hyperparameters, fingerprint-from-config, describe-driven reuse guard,
# non-blocking launches collected by cell 13.
from pathlib import Path

ABLATIONS = sorted(str(p.relative_to(REPO)) for p in (REPO / "configs" / "ablations").glob("v2_*.yaml"))
assert ABLATIONS, "no configs/ablations/v2_*.yaml found"
print("ablation ladder:", ", ".join(ABLATIONS))
print("note: configs/decider_v2_long.yaml (the 4x-budget option) is NOT in this ladder; "
      "launching it is a deliberate budget decision, never an accidental side effect.")

if SMOKE:
    print("[skip] SMOKE mode: the ladder would queue five GPUs on jobs; launch it in real mode only")
else:
    for cfg in ABLATIONS:
        name = Path(cfg).stem  # e.g. v2_lora_bev
        launch_job(f"abl-{name}", cfg, n_train=N_TRAIN, seed=0, wait=False)
    print("five ablation jobs submitted; monitor them with cell 13")


In [ ]:
# Cell 18 - cleanup / kill switch. SageMaker has NO delete-training-job API: a hanging run can
# only be STOPPED, which is terminal. This cell stops (and then records) every job this notebook
# has in an InProgress or Stopping state, both arms plus ablations; anything Completed/Failed
# stays untouched so the receipts are auditable.
for name in sorted(state["jobs"]):
    live = sm.describe_training_job(TrainingJobName=name)["TrainingJobStatus"]
    meta = state["jobs"][name]
    if live in ("InProgress", "Stopping"):
        print(f"[stop] {name} ({meta['arm']}) in state {live}")
        sm.stop_training_job(TrainingJobName=name)
        meta["status"] = "Stopping"
        meta["stopped_by"] = "cell 18 cleanup"
    elif meta.get("status") != live:
        meta["status"] = live
        print(f"[sync] {name}: recorded {live}")
save_state()
print("cleanup pass complete - re-run cell 13 once each stopped job turns Stopped and the partial "
     "artifacts get downloaded (no budget verdict for partial runs)")
